In [0]:
-- data cleaning and storing in the silver layer with the help of "COALESCE"

-- silver_sales_shipment_data

CREATE OR REPLACE TABLE main.default.silver_sales_shipment_data AS
WITH cleaned AS (
  SELECT
    `Order Id` AS order_id,
    CASE WHEN `order date (DateOrders)` = '-' THEN NULL ELSE
      COALESCE(
        TO_DATE(TRY_TO_TIMESTAMP(`order date (DateOrders)`, 'M/d/yyyy H:mm')),
        TO_DATE(TRY_TO_TIMESTAMP(`order date (DateOrders)`, 'yyyy-MM-dd H:mm:ss')),
        TO_DATE(TRY_TO_TIMESTAMP(`order date (DateOrders)`))
      )
    END AS order_date,
    CASE WHEN `shipping date (DateOrders)` = '-' THEN NULL ELSE
      COALESCE(
        TO_DATE(TRY_TO_TIMESTAMP(`shipping date (DateOrders)`, 'M/d/yyyy H:mm')),
        TO_DATE(TRY_TO_TIMESTAMP(`shipping date (DateOrders)`, 'yyyy-MM-dd H:mm:ss')),
        TO_DATE(TRY_TO_TIMESTAMP(`shipping date (DateOrders)`))
      )
    END AS shipment_date,
    `Product Id` AS product_id,
    COALESCE(`Customer Id`, -1) AS customer_id,
    COALESCE(`Customer Segment`, 'Unknown') AS customer_segment,
    COALESCE(TRY_CAST(`Sales` AS DOUBLE), 0.0) AS sales,
    COALESCE(TRY_CAST(`Order Item Quantity` AS INT), 0) AS quantity,
    COALESCE(TRY_CAST(`Order Profit Per Order` AS DOUBLE), 0.0) AS profit,
    CASE WHEN LOWER(TRIM(`Delivery Status`)) IN ('late delivery', 'late', 'yes', '1', 'true') THEN 1 ELSE 0 END AS is_late_delivery,
    COALESCE(`Shipping Mode`, 'Unknown') AS shipping_mode,
    COALESCE(`Market`, 'Unknown') AS market,
    COALESCE(`Department Name`, 'Unknown') AS department,
    COALESCE(`Class`, 'Unknown') AS class
  FROM main.default.bronze_sales_shipment_data
  WHERE `Order Id` IS NOT NULL AND `Product Id` IS NOT NULL
)
SELECT *
FROM cleaned
WHERE order_date IS NOT NULL AND shipment_date IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY order_id, product_id, order_date
  ORDER BY shipment_date DESC
) = 1;

-- null table of order date and shipment date (BAD records)

CREATE OR REPLACE TABLE main.default.silver_sales_shipment_data_rejects AS
WITH cleaned AS (
    SELECT
        `Order Id` AS order_id,
        CASE WHEN `order date (DateOrders)` = '-' THEN NULL ELSE
            COALESCE(
                TO_DATE(TRY_TO_TIMESTAMP(`order date (DateOrders)`, 'M/d/yyyy H:mm')),
                TO_DATE(TRY_TO_TIMESTAMP(`order date (DateOrders)`, 'yyyy-MM-dd H:mm:ss')),
                TO_DATE(TRY_TO_TIMESTAMP(`order date (DateOrders)`))
            )
        END AS order_date,
        CASE WHEN `shipping date (DateOrders)` = '-' THEN NULL ELSE
            COALESCE(
                TO_DATE(TRY_TO_TIMESTAMP(`shipping date (DateOrders)`, 'M/d/yyyy H:mm')),
                TO_DATE(TRY_TO_TIMESTAMP(`shipping date (DateOrders)`, 'yyyy-MM-dd H:mm:ss')),
                TO_DATE(TRY_TO_TIMESTAMP(`shipping date (DateOrders)`))
            )
        END AS shipment_date,
        `Product Id` AS product_id,
        COALESCE(`Customer Id`, -1) AS customer_id,
        COALESCE(`Customer Segment`, 'Unknown') AS customer_segment,
        COALESCE(TRY_CAST(`Sales` AS DOUBLE), 0.0) AS sales,
        COALESCE(TRY_CAST(`Order Item Quantity` AS INT), 0) AS quantity,
        COALESCE(TRY_CAST(`Order Profit Per Order` AS DOUBLE), 0.0) AS profit,
        CASE WHEN LOWER(TRIM(`Delivery Status`)) IN ('late delivery', 'late', 'yes', '1', 'true') THEN 1 ELSE 0 END AS is_late_delivery,
        COALESCE(`Shipping Mode`, 'Unknown') AS shipping_mode,
        COALESCE(`Market`, 'Unknown') AS market,
        COALESCE(`Department Name`, 'Unknown') AS department,
        COALESCE(`Class`, 'Unknown') AS class
    FROM main.default.bronze_sales_shipment_data
    WHERE `Order Id` IS NOT NULL AND `Product Id` IS NOT NULL
)
SELECT *
FROM cleaned
WHERE order_date IS NULL OR shipment_date IS NULL;

-- silver_inventory_stock_data

CREATE OR REPLACE TABLE main.default.silver_inventory_stock_data AS
WITH cleaned AS (
  SELECT
    `product id` AS product_id,
    COALESCE(TRY_CAST(`avg lead time` AS INT), 0) AS lead_time,
    COALESCE(TRY_CAST(`current stock` AS INT), 0) AS current_stock,
    COALESCE(TRY_CAST(`reorder point` AS INT), 0) AS reorder_point,
    COALESCE(TRY_CAST(`avg order qty` AS INT), 0) AS avg_order,
    COALESCE(TRY_CAST(`safety stock` AS INT), 0) AS safety_stock
  FROM main.default.bronze_inventory_stock_data
  WHERE `product id` IS NOT NULL
)
SELECT *
FROM cleaned
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY product_id
  ORDER BY current_stock DESC
) = 1;


-- silver_regional_warehouse

CREATE OR REPLACE TABLE main.default.silver_regional_warehouse AS
WITH cleaned AS (
  SELECT
    `warehouse_id`,
    COALESCE(`region_type`, 'Unknown') AS region_type,
    COALESCE(TRY_CAST(`has_wms` AS INT), 0) AS has_wms,
    COALESCE(TRY_CAST(`capacity_utilisation_pct` AS DOUBLE), 0.0) AS capacity_utilisation_pct
  FROM main.default.bronze_warehouse_regional_warehouse
  WHERE `warehouse_id` IS NOT NULL
)
SELECT *
FROM cleaned
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY warehouse_id
  ORDER BY capacity_utilisation_pct DESC
) = 1;


-- silver_district_store

CREATE OR REPLACE TABLE main.default.silver_district_store AS
WITH cleaned AS (
  SELECT
    `warehouse_id`,
    COALESCE(`region_type`, 'Unknown') AS region_type,
    COALESCE(TRY_CAST(`has_wms` AS INT), 0) AS has_wms,
    COALESCE(TRY_CAST(`warehouse_size_sqm` AS INT), 0) AS capacity
  FROM main.default.bronze_warehouse_district_store
  WHERE `warehouse_id` IS NOT NULL
)
SELECT *
FROM cleaned
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY warehouse_id
  ORDER BY capacity DESC
) = 1;


-- silver_national_medical_store

CREATE OR REPLACE TABLE main.default.silver_national_medical_store AS
WITH cleaned AS (
  SELECT
    `warehouse_id`,
    COALESCE(`region_type`, 'Unknown') AS region_type,
    COALESCE(TRY_CAST(`has_wms` AS INT), 0) AS has_wms,
    COALESCE(TRY_CAST(`warehouse_size_sqm` AS INT), 0) AS capacity
  FROM main.default.bronze_warehouse_national_central_medical_store
  WHERE `warehouse_id` IS NOT NULL
)
SELECT *
FROM cleaned
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY warehouse_id
  ORDER BY capacity DESC
) = 1;


-- silver_supply_chain_data

CREATE OR REPLACE TABLE main.default.silver_supply_chain_data AS
WITH cleaned AS (
  SELECT
    `SKU` AS sku,
    `Product type` AS product_type,
    COALESCE(TRY_CAST(`Price` AS DOUBLE), 0.0) AS price,
    COALESCE(TRY_CAST(`Availability` AS INT), 0) AS availability,
    COALESCE(TRY_CAST(`Number of products sold` AS INT), 0) AS number_of_products_sold,
    COALESCE(TRY_CAST(`Revenue generated` AS DOUBLE), 0.0) AS revenue,
    COALESCE(`Customer demographics`, 'Unknown') AS customer_demographics,
    COALESCE(TRY_CAST(`Stock levels` AS INT), 0) AS stock_levels,
    COALESCE(TRY_CAST(`Lead times` AS INT), 0) AS lead_times,
    COALESCE(TRY_CAST(`Order quantities` AS INT), 0) AS order_quantities,
    COALESCE(TRY_CAST(`Shipping times` AS INT), 0) AS shipping_times,
    COALESCE(`Shipping carriers`, 'Unknown') AS shipping_carriers,
    COALESCE(TRY_CAST(`Shipping costs` AS DOUBLE), 0.0) AS shipping_costs,
    COALESCE(`Supplier name`, 'Unknown') AS supplier_name,
    COALESCE(`Location`, 'Unknown') AS location,
    COALESCE(TRY_CAST(`Defect rates` AS DOUBLE), 0.0) AS defect_rates
  FROM main.default.bronze_supply_chain_data
  WHERE `SKU` IS NOT NULL
)
SELECT *
FROM cleaned
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY sku
  ORDER BY revenue DESC
) = 1;


-- silver_retail_store_inventory

CREATE OR REPLACE TABLE main.default.silver_retail_store_inventory AS
WITH cleaned AS (
  SELECT
    COALESCE(
      TO_DATE(TRY_TO_TIMESTAMP(`Date`, 'M/d/yyyy H:mm')),
      TO_DATE(TRY_TO_TIMESTAMP(`Date`, 'yyyy-MM-dd H:mm:ss')),
      TO_DATE(TRY_TO_TIMESTAMP(`Date`))
    ) AS date,
    `Store ID` AS store_id,
    `Product ID` AS product_id,
    COALESCE(TRY_CAST(`Units Sold` AS DOUBLE), 0.0) AS sales,
    COALESCE(TRY_CAST(`Inventory Level` AS INT), 0) AS stock
  FROM main.default.bronze_retail_store_inventory
  WHERE `Store ID` IS NOT NULL AND `Product ID` IS NOT NULL AND `Date` IS NOT NULL
)
SELECT *
FROM cleaned
WHERE date IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY store_id, product_id, date
  ORDER BY sales DESC
) = 1;


SHOW TABLES IN main.default;
SELECT * FROM main.default.silver_sales_shipment_data_rejects;
